# Main Paper Figures

Clean, manuscript-facing figure notebook. Each section corresponds to one candidate main-text figure and writes a complete multi-panel figure to `notebooks/figures/main/`. The exploratory notebook remains useful for debugging individual panels.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter

ROOT = Path.cwd()
while not (ROOT / "results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))

import paper_plot_data_loaders as plot_loaders
importlib.reload(plot_loaders)
from paper_plot_data_loaders import (
    load_reinvent_liability,
    load_guacamol_liability,
    load_guacamol_qed,
    PAPER_SEEDS_10,
)

RESULTS = ROOT / "results"
FIG_DIR = ROOT / "notebooks" / "figures" / "main"
DATA_DIR = FIG_DIR / "plotting_data"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.labelsize": 9.5,
    "axes.titlesize": 10.5,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.edgecolor": "#111111",
    "axes.linewidth": 0.9,
    "xtick.color": "#111111",
    "ytick.color": "#111111",
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

METHOD_COLORS = {
    "base": "#4B5563",
    "random_neon": "#A3A3A3",
    "positive": "#EE9B00",
    "neon": "#1A759F",
    "positive_corrected_ne": "#2D6A4F",
}
NE_LAMBDA_COLORS = {0.10: "#184E77", 0.25: "#1E6091", 0.50: "#1A759F", 0.75: "#168AAD", 1.00: "#34A0A4"}

def style_ax(ax):
    ax.grid(False)
    ax.tick_params(axis="both", which="major", bottom=True, left=True, length=3.5, width=0.8, color="#111111", labelcolor="#111111")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#111111")
        spine.set_linewidth(0.9)

def save_figure(fig, name: str):
    for suffix in ("png", "svg", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{suffix}", bbox_inches="tight")
    print(FIG_DIR / f"{name}.png")

def label_subplots(fig, labels=None, x=-0.12, y=1.045, axes=None):
    if axes is None:
        axes = [ax for ax in fig.axes if ax.get_visible()]
    if labels is None:
        labels = [chr(ord("a") + i) for i in range(len(axes))]
    for ax, label in zip(axes, labels):
        ax.text(
            x,
            y,
            label,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontweight="bold",
            fontsize=11.0,
            color="#111111",
            clip_on=False,
        )

def paired_reference_boxplot(
    ax, frame, *, model_col, value_col, reference_col, model_order,
    model_labels, model_colors, reference_order, reference_labels,
):
    positions = np.arange(len(model_order), dtype=float)
    for reference, offset, hatch in zip(reference_order, (-0.17, 0.17), (None, "///")):
        for model_index, model in enumerate(model_order):
            values = frame.loc[
                frame[model_col].eq(model) & frame[reference_col].eq(reference), value_col
            ].dropna().to_numpy(dtype=float)
            if not len(values):
                continue
            artists = ax.boxplot(
                [values], positions=[positions[model_index] + offset], widths=0.28,
                patch_artist=True, showfliers=False,
                medianprops={"color": "#111111", "linewidth": 1.0},
                boxprops={"edgecolor": "#111111", "linewidth": 0.8},
                whiskerprops={"color": "#111111", "linewidth": 0.8},
                capprops={"color": "#111111", "linewidth": 0.8},
            )
            box = artists["boxes"][0]
            box.set_facecolor(model_colors[model])
            box.set_alpha(0.90 if reference == reference_order[0] else 0.62)
            if hatch:
                box.set_hatch(hatch)
    ax.set_xticks(positions, [model_labels[model] for model in model_order], rotation=25, ha="right")
    ax.legend(
        handles=[
            Patch(facecolor="white", edgecolor="#111111", label=reference_labels[reference_order[0]]),
            Patch(facecolor="white", edgecolor="#111111", hatch="///", label=reference_labels[reference_order[1]]),
        ],
        frameon=False, loc="upper left",
    )
    style_ax(ax)



def mean_ci(df: pd.DataFrame, group_cols: list[str], value_col: str) -> pd.DataFrame:
    out = df.groupby(group_cols, observed=True)[value_col].agg(mean="mean", std="std", n="count").reset_index()
    out["sem"] = out["std"] / np.sqrt(out["n"].clip(lower=1))
    out["t_critical"] = out["n"].map(lambda n: float(stats.t.ppf(0.975, n - 1)) if n > 1 else np.nan)
    out["ci95"] = out["t_critical"] * out["sem"]
    return out


## Figure 1: QED Control Experiment

Control/caveat task: when high-QED positive examples are abundant and easy to select, positive fine-tuning is a strong comparator. Endpoint panels compare selected methods at λ=1.0; lambda-response panels show whether NE improves QED gradually or trades off validity.


In [ ]:
qed = load_guacamol_qed()
qed["model"] = qed["model"].astype(str)

QED_SELECTED_MODELS = ["base", "random_neon_lambda_1.0", "positive", "neon_lambda_1.0"]
QED_LABELS = {
    "base": "Base",
    "random_neon_lambda_1.0": "Random NE 1.0",
    "positive": "Positive FT",
    "neon_lambda_1.0": "NE 1.0",
}
QED_COLORS = {
    "base": METHOD_COLORS["base"],
    "random_neon_lambda_1.0": METHOD_COLORS["random_neon"],
    "positive": METHOD_COLORS["positive"],
    "neon_lambda_1.0": NE_LAMBDA_COLORS[1.00],
}
QED_ARCH_ORDER = ["RNN", "Transformer"]
QED_METRICS = [
    ("qed_mean", "Mean QED"),
    ("qed_ge_0.9_fraction", "QED >= 0.9"),
    ("valid_fraction", "Validity"),
]

qed_endpoint = qed[qed["model"].isin(QED_SELECTED_MODELS)].copy()
qed_endpoint["method_label"] = qed_endpoint["model"].map(QED_LABELS)
qed_endpoint["method_label"] = pd.Categorical(
    qed_endpoint["method_label"],
    [QED_LABELS[m] for m in QED_SELECTED_MODELS],
    ordered=True,
)

lambda_rows = []
for model in sorted(qed["model"].unique()):
    if model.startswith("neon_lambda_") or model.startswith("random_neon_lambda_"):
        try:
            lam = float(model.rsplit("_", 1)[-1])
        except ValueError:
            continue
        family = "NE" if model.startswith("neon_lambda_") else "Random NE"
        lambda_rows.append({"model": model, "lambda": lam, "family": family})
qed_lambda_meta = pd.DataFrame(lambda_rows)
qed_lambda = qed.merge(qed_lambda_meta, on="model", how="inner")
qed_reference = (
    qed[qed["model"].isin(["base", "positive"])]
    .groupby(["architecture", "model"], observed=True)[[metric for metric, _ in QED_METRICS]]
    .mean()
    .reset_index()
)
lambda_summaries = {
    metric: mean_ci(qed_lambda, ["architecture", "family", "lambda"], metric)
    for metric, _ in QED_METRICS
}

fig, axes = plt.subplots(4, 3, figsize=(11.5, 10.2), sharex="row", constrained_layout=True)
family_colors = {"NE": NE_LAMBDA_COLORS[0.75], "Random NE": METHOD_COLORS["random_neon"]}

for arch_index, architecture in enumerate(QED_ARCH_ORDER):
    arch_endpoint = qed_endpoint[qed_endpoint["architecture"].eq(architecture)]
    for col, (metric, ylabel) in enumerate(QED_METRICS):
        ax = axes[arch_index, col]
        sns.boxplot(
            data=arch_endpoint,
            x="method_label",
            y=metric,
            order=[QED_LABELS[m] for m in QED_SELECTED_MODELS],
            palette=[QED_COLORS[m] for m in QED_SELECTED_MODELS],
            width=0.58,
            linewidth=0.5,
            fliersize=0,
            ax=ax,
        )
        ax.set_title(f"{architecture}: {ylabel}", loc="left", fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel if col == 0 else "")
        if metric.endswith("fraction"):
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        style_ax(ax)

for arch_index, architecture in enumerate(QED_ARCH_ORDER, start=2):
    refs = qed_reference[qed_reference["architecture"].eq(architecture)]
    for col, (metric, ylabel) in enumerate(QED_METRICS):
        ax = axes[arch_index, col]
        summary = lambda_summaries[metric]
        part = summary[summary["architecture"].eq(architecture)]
        for family in ["Random NE", "NE"]:
            fam = part[part["family"].eq(family)].sort_values("lambda")
            ax.plot(fam["lambda"], fam["mean"], marker="o", lw=1.8, color=family_colors[family], label=family)
            ax.fill_between(
                fam["lambda"].to_numpy(float),
                (fam["mean"] - fam["ci95"]).to_numpy(float),
                (fam["mean"] + fam["ci95"]).to_numpy(float),
                color=family_colors[family],
                alpha=0.16,
                linewidth=0,
            )
        for model, color, label in [("base", METHOD_COLORS["base"], "Base"), ("positive", METHOD_COLORS["positive"], "Positive FT")]:
            ref = refs[refs["model"].eq(model)]
            if not ref.empty:
                ax.axhline(float(ref[metric].iloc[0]), color=color, lw=1.1, ls="--", alpha=0.75, label=label if (arch_index == 2 and col == 0) else None)
        ax.set_title(f"{architecture}: {ylabel} lambda response", loc="left", fontweight="bold")
        ax.set_xlabel("λ")
        ax.set_ylabel(ylabel if col == 0 else "")
        if metric.endswith("fraction"):
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        style_ax(ax)

axes[2, 0].legend(frameon=False, ncol=2, loc="best")
label_subplots(fig)
save_figure(fig, "figure_1_qed_control")
plt.show()

qed_endpoint.to_csv(DATA_DIR / "figure_1_qed_endpoint.csv", index=False)
qed_lambda.to_csv(DATA_DIR / "figure_1_qed_lambda_response.csv", index=False)


## Figure 2: Transformer Scope Ablation

Main claim: full-model NE can over-steer a Transformer and collapse validity. Restricting the extrapolated update to the final Transformer block plus output head preserves validity while retaining liability removal, motivating the scoped Transformer setting used in later GuacaMol liability panels.


In [ ]:
SCOPE_OBJECTIVE = "Metal-binding motif"
SCOPE_METRIC = "chelator_hit_fraction"
SCOPE_DIR = RESULTS / "objectives"

SCOPE_LABELS = {
    "full": "Full model",
    "no_embeddings_layernorm": "No embeddings / layer norm",
    "last2_blocks_output": "Last 2 blocks + output",
    "last_block_output": "Last block + output",
    "output": "Output only",
}
SCOPE_ORDER = ["full", "no_embeddings_layernorm", "last2_blocks_output", "last_block_output", "output"]
SCOPE_COLORS = {
    "full": NE_LAMBDA_COLORS[0.75],
    "no_embeddings_layernorm": "#B5E48C",
    "last2_blocks_output": "#99D98C",
    "last_block_output": "#52B69A",
    "output": "#D9ED92",
}


def _read_objective_csv(folder: str) -> pd.DataFrame:
    path = SCOPE_DIR / folder / "objective_metrics.csv"
    frame = pd.read_csv(path).copy()
    frame["source_folder"] = folder
    return frame


def _parse_scoped_ne_model(model: str):
    model = str(model)
    if model.startswith("random_") or not model.startswith("neon_"):
        return None
    text = model.removeprefix("neon_")
    if text.startswith("lambda_"):
        return {"scope": "full", "lambda": float(text.removeprefix("lambda_"))}
    if "_lambda_" not in text:
        return None
    scope, lam_text = text.rsplit("_lambda_", 1)
    if scope in SCOPE_LABELS:
        return {"scope": scope, "lambda": float(lam_text)}
    return None

# Clean development experiment: every scope-lambda combination uses the same three seeds.
SCOPE_DEVELOPMENT_SEEDS = [5, 7, 11]
SCOPE_LAMBDAS = [0.1, 0.25, 0.5, 0.75, 1.0]
scope_response = _read_objective_csv("guacamol_transformer_chelator_scope_development")
parsed = scope_response["model"].map(_parse_scoped_ne_model).apply(
    lambda x: pd.Series(x) if x else pd.Series(dtype=float)
)
scope_response = pd.concat([scope_response, parsed], axis=1)
scope_response = scope_response[
    scope_response["scope"].isin(SCOPE_ORDER)
    & scope_response["seed"].isin(SCOPE_DEVELOPMENT_SEEDS)
    & scope_response["lambda"].isin(SCOPE_LAMBDAS)
].copy()
scope_response["scope_label"] = scope_response["scope"].map(SCOPE_LABELS)

cell_counts = scope_response.groupby(["scope", "lambda"], observed=True)["seed"].nunique()
expected_cells = pd.MultiIndex.from_product([SCOPE_ORDER, SCOPE_LAMBDAS], names=["scope", "lambda"])
cell_counts = cell_counts.reindex(expected_cells, fill_value=0)
if not cell_counts.eq(len(SCOPE_DEVELOPMENT_SEEDS)).all():
    missing = cell_counts[cell_counts.ne(len(SCOPE_DEVELOPMENT_SEEDS))]
    raise ValueError(f"Incomplete clean scope experiment; seed counts by scope/lambda:\n{missing}")

scope_response_summary = (
    scope_response.groupby(["scope", "scope_label", "lambda"], observed=True)[
        [SCOPE_METRIC, "valid_fraction"]
    ]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.1), constrained_layout=True)

screen_lambda = 1.0
screen_plot = scope_response[np.isclose(scope_response["lambda"], screen_lambda)].copy()
for col, (metric, ylabel) in enumerate([(SCOPE_METRIC, "Metal-binding-motif hit rate"), ("valid_fraction", "Validity")]):
    ax = axes[0, col]
    for scope_index, scope in enumerate(SCOPE_ORDER):
        values = screen_plot.loc[screen_plot["scope"].eq(scope), metric].dropna().to_numpy(float)
        offsets = np.linspace(-0.045, 0.045, len(values)) if len(values) > 1 else np.zeros(len(values))
        ax.scatter(scope_index + offsets, values, s=28, color=SCOPE_COLORS[scope], edgecolor="#111111", linewidth=0.45, zorder=3)
        if len(values):
            ax.plot([scope_index - 0.13, scope_index + 0.13], [values.mean(), values.mean()], color="#111111", lw=1.1, zorder=4)
    ax.set_title(f"Scope screen at λ={screen_lambda:g}", loc="left", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xticks(range(len(SCOPE_ORDER)), [SCOPE_LABELS[s] for s in SCOPE_ORDER], rotation=45, ha="right")
    style_ax(ax)

for ax, metric, ylabel in [
    (axes[1, 0], SCOPE_METRIC, "Metal-binding-motif hit rate"),
    (axes[1, 1], "valid_fraction", "Validity"),
]:
    for scope in ["full", "last_block_output"]:
        raw_scope = scope_response[scope_response["scope"].eq(scope)]
        for _, seed_data in raw_scope.groupby("seed", observed=True):
            seed_data = seed_data.sort_values("lambda")
            ax.plot(seed_data["lambda"], seed_data[metric], marker="o", ms=2.8, lw=0.8, color=SCOPE_COLORS[scope], alpha=0.28, zorder=1)
        part = scope_response_summary[scope_response_summary["scope"].eq(scope)].sort_values("lambda")
        ax.plot(part["lambda"], part[metric], marker="o", lw=2.0, color=SCOPE_COLORS[scope], label=SCOPE_LABELS[scope], zorder=3)
    ax.set_title("Lambda response", loc="left", fontweight="bold")
    ax.set_xlabel("λ")
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xticks(SCOPE_LAMBDAS)
    style_ax(ax)

axes[1, 1].legend(frameon=False, loc="best")
label_subplots(fig)
save_figure(fig, "figure_2_transformer_scope_ablation")
plt.show()

screen_plot.to_csv(DATA_DIR / "figure_2_transformer_scope_screen.csv", index=False)
scope_response.to_csv(DATA_DIR / "figure_2_transformer_scope_response.csv", index=False)


## Figure 3: Custom GuacaMol Liability Objectives

Main claim: after choosing the scoped Transformer update, NE transfers across custom RNN and Transformer generators for liability-specific removal. Validity panels are shown directly under the hit-rate panels to distinguish these liability settings from the QED control where larger updates can trade objective gain for degraded generation quality.


In [ ]:
guacamol = load_guacamol_liability()
GUACAMOL_OBJECTIVE_ORDER = ["Reactive", "Metal-binding motif", "Charged motif", "Assay interference"]
GUACAMOL_MODELS = ["base", "random_neon_1", "positive", "neon_1"]
GUACAMOL_LABELS = {"base": "Base", "random_neon_1": "Random NE 1.0", "positive": "Positive FT", "neon_1": "NE 1.0"}
GUACAMOL_COLORS = {"base": METHOD_COLORS["base"], "random_neon_1": METHOD_COLORS["random_neon"], "positive": METHOD_COLORS["positive"], "neon_1": NE_LAMBDA_COLORS[1.00]}
GUACAMOL_FAMILY_COLORS = {"Random NE": METHOD_COLORS["random_neon"], "NE": NE_LAMBDA_COLORS[0.75]}


def normalize_guacamol_model(model: str) -> str | None:
    model = str(model)
    if model == "base":
        return "base"
    if model == "positive":
        return "positive"
    if model in {"neon_lambda_1", "neon_lambda_1.0", "neon_last_block_output_lambda_1.0", "neon_last_block_output_lambda_1"}:
        return "neon_1"
    if model in {"random_neon_lambda_1", "random_neon_lambda_1.0", "random_neon_last_block_output_lambda_1.0", "random_neon_last_block_output_lambda_1"}:
        return "random_neon_1"
    return None


def parse_guacamol_lambda_model(model: str):
    model = str(model)
    prefixes = [
        ("random_neon_last_block_output_lambda_", "Random NE"),
        ("random_neon_lambda_", "Random NE"),
        ("neon_last_block_output_lambda_", "NE"),
        ("neon_lambda_", "NE"),
    ]
    for prefix, family in prefixes:
        if model.startswith(prefix):
            try:
                return {"family": family, "lambda": float(model.removeprefix(prefix))}
            except ValueError:
                return None
    return None


def load_guacamol_chelator_structure(architecture: str) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    """Load representative metal-binding-motif-removal structural audit tables."""
    if architecture == "RNN":
        diversity_path = RESULTS / "objectives/guacamol_rnn_chelator_removal/analysis/diversity_selected_10seed/diversity_metrics.csv"
        distance_path = RESULTS / "objectives/guacamol_rnn_chelator_removal/analysis/distribution_distance_fcd_base_filtered_selected/distribution_distance_metrics.csv"
        liability_free_reference = "Liability-free base"
    elif architecture == "Transformer":
        diversity_path = RESULTS / "objectives/guacamol_transformer_chelator_lastblock_final/analysis/diversity_selected_10seed/diversity_metrics.csv"
        distance_path = RESULTS / "objectives/guacamol_transformer_chelator_lastblock_final/analysis/distribution_distance_fcd_10seed/distribution_distance_metrics.csv"
        # The transformer audit stores the selected liability-free fine-tuning set as `good`.
        liability_free_reference = "Liability-free reference"
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    diversity = pd.read_csv(diversity_path).copy()
    distance = pd.read_csv(distance_path).copy()
    diversity = diversity[diversity["seed"].isin(PAPER_SEEDS_10)].copy()
    distance = distance[distance["seed"].isin(PAPER_SEEDS_10)].copy()
    for frame in [diversity, distance]:
        frame["method_plot"] = frame["model"].map(normalize_guacamol_model)
        frame["method_label"] = frame["method_plot"].map(GUACAMOL_LABELS)
        frame["method_label"] = pd.Categorical(
            frame["method_label"],
            [GUACAMOL_LABELS[m] for m in GUACAMOL_MODELS],
            ordered=True,
        )
    diversity = diversity[diversity["method_plot"].isin(GUACAMOL_MODELS)].copy()
    distance = distance[distance["method_plot"].isin(GUACAMOL_MODELS)].copy()
    distance["reference_label"] = distance["reference"].map(
        {
            "base": "Base",
            "base_filtered": "Liability-free base",
            "good": "Liability-free reference",
            "bad": "Bad set",
        }
    )
    return diversity, distance, liability_free_reference


guacamol["method_plot"] = guacamol["model"].map(normalize_guacamol_model)
g_endpoint = guacamol[guacamol["method_plot"].isin(GUACAMOL_MODELS)].copy()
g_endpoint["method_label"] = g_endpoint["method_plot"].map(GUACAMOL_LABELS)
g_endpoint["method_label"] = pd.Categorical(g_endpoint["method_label"], [GUACAMOL_LABELS[m] for m in GUACAMOL_MODELS], ordered=True)
g_endpoint["objective_label"] = pd.Categorical(g_endpoint["objective_label"], GUACAMOL_OBJECTIVE_ORDER, ordered=True)

lambda_meta = guacamol["model"].map(parse_guacamol_lambda_model).apply(lambda x: pd.Series(x) if x else pd.Series(dtype=float))
g_lambda = pd.concat([guacamol.copy(), lambda_meta], axis=1)
g_lambda = g_lambda[g_lambda["family"].isin(["Random NE", "NE"])].copy()
g_lambda["objective_label"] = pd.Categorical(g_lambda["objective_label"], GUACAMOL_OBJECTIVE_ORDER, ordered=True)
g_valid_summary = mean_ci(g_lambda, ["architecture", "objective_label", "family", "lambda"], "valid_fraction")
g_reference = (
    guacamol[guacamol["model"].isin(["base", "positive"])]
    .groupby(["architecture", "objective_label", "model"], observed=True)["valid_fraction"]
    .mean()
    .reset_index()
)

g_structure = {
    arch: load_guacamol_chelator_structure(arch)
    for arch in ["RNN", "Transformer"]
}

fig, axes = plt.subplots(6, 4, figsize=(13.0, 15.0), sharey=False, constrained_layout=True)
g_grid = axes[0, 0].get_gridspec()
g_structure_fcd_axes = {}
for structure_row, architecture in [(2, "RNN"), (5, "Transformer")]:
    axes[structure_row, 2].remove()
    axes[structure_row, 3].remove()
    g_structure_fcd_axes[architecture] = fig.add_subplot(g_grid[structure_row, 2:4])
g_panel_axes = []
row_specs = [
    ("RNN", "endpoint"),
    ("RNN", "validity_response"),
    ("RNN", "structure"),
    ("Transformer", "endpoint"),
    ("Transformer", "validity_response"),
    ("Transformer", "structure"),
]
for row, (arch, panel_type) in enumerate(row_specs):
    for col, objective in enumerate(GUACAMOL_OBJECTIVE_ORDER):
        if panel_type == "structure" and col >= 2:
            if col == 3:
                continue
            ax = g_structure_fcd_axes[arch]
        else:
            ax = axes[row, col]
        g_panel_axes.append(ax)
        if panel_type == "endpoint":
            df = g_endpoint[g_endpoint["architecture"].eq(arch) & g_endpoint["objective_label"].eq(objective)]
            sns.boxplot(
                data=df,
                x="method_label",
                y="target_hit_fraction",
                order=[GUACAMOL_LABELS[m] for m in GUACAMOL_MODELS],
                palette=[GUACAMOL_COLORS[m] for m in GUACAMOL_MODELS],
                width=0.58,
                linewidth=0.5,
                fliersize=0,
                ax=ax,
            )
            ax.set_title(objective if row in {0, 3} else "", loc="left", fontweight="bold")
            ax.set_xlabel("")
            ax.set_ylabel(f"{arch}\nTarget hit rate" if col == 0 else "")
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        elif panel_type == "validity_response":
            part = g_valid_summary[
                g_valid_summary["architecture"].eq(arch)
                & g_valid_summary["objective_label"].astype(str).eq(objective)
            ]
            for family in ["Random NE", "NE"]:
                fam = part[part["family"].eq(family)].sort_values("lambda")
                if fam.empty:
                    continue
                ax.plot(fam["lambda"], fam["mean"], marker="o", lw=1.8, color=GUACAMOL_FAMILY_COLORS[family], label=family)
                ax.fill_between(
                    fam["lambda"].to_numpy(float),
                    (fam["mean"] - fam["ci95"]).to_numpy(float),
                    (fam["mean"] + fam["ci95"]).to_numpy(float),
                    color=GUACAMOL_FAMILY_COLORS[family],
                    alpha=0.16,
                    linewidth=0,
                )
            refs = g_reference[
                g_reference["architecture"].eq(arch)
                & g_reference["objective_label"].astype(str).eq(objective)
            ]
            for model, color, label in [("base", METHOD_COLORS["base"], "Base"), ("positive", METHOD_COLORS["positive"], "Positive FT")]:
                ref = refs[refs["model"].eq(model)]
                if not ref.empty:
                    ax.axhline(float(ref["valid_fraction"].iloc[0]), color=color, lw=1.1, ls="--", alpha=0.75, label=label if (row == 1 and col == 0) else None)
            ax.set_xlabel("λ")
            ax.set_ylabel(f"{arch}\nValidity" if col == 0 else "")
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
        else:
            diversity, distance, liability_free_reference = g_structure[arch]
            if col < 2:
                structure_specs = [
                    (diversity, "unique_scaffold_fraction", "Unique scaffold fraction", "percent"),
                    (diversity, "pairwise_tanimoto_distance_mean", "Pairwise Tanimoto distance", "decimal"),
                ]
                frame, metric, title, scale = structure_specs[col]
                sns.boxplot(
                    data=frame,
                    x="method_label",
                    y=metric,
                    order=[GUACAMOL_LABELS[m] for m in GUACAMOL_MODELS],
                    palette=[GUACAMOL_COLORS[m] for m in GUACAMOL_MODELS],
                    width=0.58,
                    linewidth=0.5,
                    fliersize=0,
                    ax=ax,
                )
                if scale == "percent":
                    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
                    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
                elif scale == "decimal":
                    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            else:
                title = "FCD to reference"
                paired_reference_boxplot(
                    ax, distance, model_col="method_plot", value_col="fcd", reference_col="reference_label",
                    model_order=GUACAMOL_MODELS, model_labels=GUACAMOL_LABELS, model_colors=GUACAMOL_COLORS,
                    reference_order=["Base", liability_free_reference],
                    reference_labels={"Base": "Raw base", liability_free_reference: liability_free_reference},
                )
            ax.set_title(title, loc="left", fontweight="bold")
            ax.set_xlabel("")
            structure_ylabels = {
                0: f"{arch}\nScaffold fraction",
                1: "Tanimoto distance",
                2: "FCD",
            }
            ax.set_ylabel(structure_ylabels[col])
            if col < 2:
                ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        style_ax(ax)


g_handles, g_labels = axes[1, 0].get_legend_handles_labels()
g_legend = dict(zip(g_labels, g_handles))
g_legend_order = ["Base", "Random NE", "Positive FT", "NE"]
fig.legend(
    [g_legend[label] for label in g_legend_order], g_legend_order,
    loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=4, frameon=False,
)
label_subplots(fig, axes=g_panel_axes)
save_figure(fig, "figure_3_guacamol_liability_endpoint")
plt.show()
g_endpoint.to_csv(DATA_DIR / "figure_3_guacamol_liability_endpoint.csv", index=False)
g_lambda.to_csv(DATA_DIR / "figure_3_guacamol_liability_lambda_response.csv", index=False)
for arch, (diversity, distance, _) in g_structure.items():
    tag = arch.lower()
    diversity.to_csv(DATA_DIR / f"figure_3_guacamol_{tag}_chelator_internal_diversity.csv", index=False)
    distance.to_csv(DATA_DIR / f"figure_3_guacamol_{tag}_chelator_distribution_distance.csv", index=False)


## Figure 4: REINVENT Prior Liability Removal

Main claim: in a published pretrained generator, NE reduces specific liability classes and improves fixed-budget usable yield relative to base/random controls. Endpoint panels compare selected methods at λ=1.0 and show practical fixed-budget `usable_yield`. Lambda-response panels show that the effect scales with the NE step size and include valid output yield as a sampling-budget guardrail.

`usable_yield = N_valid_unique_novel_liability_free / N_sampled`; the lambda-response guardrail uses `output_yield_fraction = N_valid / N_sampled`.

In [ ]:
reinvent = load_reinvent_liability()
REINVENT_OBJECTIVE_ORDER = ["Reactive", "Metal-binding motif", "Charged motif", "Assay interference"]
REINVENT_SELECTED_MODELS = ["base", "random_neon_lambda_1", "positive", "neon_lambda_1"]
REINVENT_LABELS = {
    "base": "Base",
    "random_neon_lambda_1": "Random NE 1.0",
    "positive": "Positive FT",
    "neon_lambda_1": "NE 1.0",
}
REINVENT_COLORS = {
    "base": METHOD_COLORS["base"],
    "random_neon_lambda_1": METHOD_COLORS["random_neon"],
    "positive": METHOD_COLORS["positive"],
    "neon_lambda_1": NE_LAMBDA_COLORS[1.00],
}
endpoint_reinvent = reinvent[reinvent["model"].isin(REINVENT_SELECTED_MODELS)].copy()
endpoint_reinvent["method_label"] = endpoint_reinvent["model"].map(REINVENT_LABELS)
endpoint_reinvent["method_label"] = pd.Categorical(endpoint_reinvent["method_label"], [REINVENT_LABELS[m] for m in REINVENT_SELECTED_MODELS], ordered=True)
endpoint_reinvent["objective_label"] = pd.Categorical(endpoint_reinvent["objective_label"], REINVENT_OBJECTIVE_ORDER, ordered=True)

lambda_models = []
for model in reinvent["model"].astype(str).unique():
    if model.startswith("neon_lambda_") or model.startswith("random_neon_lambda_"):
        try:
            lam = float(model.rsplit("_", 1)[-1])
        except ValueError:
            continue
        family = "NE" if model.startswith("neon_lambda_") else "Random NE"
        lambda_models.append((model, family, lam))
lambda_meta = pd.DataFrame(lambda_models, columns=["model", "family", "lambda"])
lambda_data = reinvent.merge(lambda_meta, on="model", how="inner")
lambda_summary = mean_ci(lambda_data, ["objective_label", "family", "lambda"], "target_hit_fraction")
lambda_output_summary = mean_ci(lambda_data, ["objective_label", "family", "lambda"], "output_yield_fraction")
reference_means = (
    endpoint_reinvent
    .groupby(["objective_label", "model"], observed=True)[["target_hit_fraction", "output_yield_fraction"]]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(5, 4, figsize=(13.2, 12.8), sharex="row", constrained_layout=True)
reinvent_grid = axes[0, 0].get_gridspec()
axes[4, 2].remove()
axes[4, 3].remove()
reinvent_fcd_ax = fig.add_subplot(reinvent_grid[4, 2:4])
family_colors = {"NE": NE_LAMBDA_COLORS[0.75], "Random NE": METHOD_COLORS["random_neon"]}
endpoint_rows = [("target_hit_fraction", "Target liability hit rate"), ("usable_yield", "Usable yield")]
lambda_rows = [(lambda_summary, "target_hit_fraction", "Target liability hit rate"), (lambda_output_summary, "output_yield_fraction", "Valid output yield")]

for col, objective in enumerate(REINVENT_OBJECTIVE_ORDER):
    df = endpoint_reinvent[endpoint_reinvent["objective_label"].eq(objective)]
    for row, (metric, ylabel) in enumerate(endpoint_rows):
        ax = axes[row, col]
        sns.boxplot(
            data=df,
            x="method_label",
            y=metric,
            order=[REINVENT_LABELS[m] for m in REINVENT_SELECTED_MODELS],
            palette=[REINVENT_COLORS[m] for m in REINVENT_SELECTED_MODELS],
            width=0.58,
            linewidth=0.5,
            fliersize=0,
            ax=ax,
        )
        ax.set_title(objective if row == 0 else "", loc="left", fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel(ylabel if col == 0 else "")
        if metric in {"output_yield_fraction", "usable_yield"}:
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
        else:
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        style_ax(ax)

    for row_offset, (summary, metric, ylabel) in enumerate(lambda_rows, start=2):
        ax = axes[row_offset, col]
        part = summary[summary["objective_label"].astype(str).eq(objective)]
        for family in ["Random NE", "NE"]:
            f = part[part["family"].eq(family)].sort_values("lambda")
            ax.plot(f["lambda"], f["mean"], marker="o", lw=1.8, color=family_colors[family], label=family)
            ax.fill_between(
                f["lambda"].to_numpy(float),
                (f["mean"] - f["ci95"]).to_numpy(float),
                (f["mean"] + f["ci95"]).to_numpy(float),
                color=family_colors[family],
                alpha=0.16,
                linewidth=0,
            )
        refs = reference_means[reference_means["objective_label"].astype(str).eq(objective)]
        for model, color, label in [("base", METHOD_COLORS["base"], "Base"), ("positive", METHOD_COLORS["positive"], "Positive FT")]:
            ref = refs[refs["model"].eq(model)]
            if not ref.empty:
                ax.axhline(float(ref[metric].iloc[0]), color=color, lw=1.1, ls="--", alpha=0.75, label=label if col == 0 else None)
        ax.set_xlabel("λ" if row_offset == 3 else "")
        ax.set_ylabel(ylabel if col == 0 else "")
        if metric == "output_yield_fraction":
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
        else:
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        style_ax(ax)


# Representative structural-diversity audit for metal-binding-motif removal.
reinvent_diversity = pd.read_csv(
    RESULTS / "external/reinvent4/chelator_replicates/analysis/diversity/diversity_metrics.csv"
).copy()
reinvent_distance = pd.read_csv(
    RESULTS / "external/reinvent4/chelator_replicates/analysis/distribution_distance_fcd/distribution_distance_metrics.csv"
).copy()
reinvent_diversity = reinvent_diversity[reinvent_diversity["seed"].isin(PAPER_SEEDS_10)].copy()
reinvent_distance = reinvent_distance[reinvent_distance["seed"].isin(PAPER_SEEDS_10)].copy()
for frame in [reinvent_diversity, reinvent_distance]:
    frame["method_label"] = frame["model"].map(REINVENT_LABELS)
    frame["method_label"] = pd.Categorical(frame["method_label"], [REINVENT_LABELS[m] for m in REINVENT_SELECTED_MODELS], ordered=True)
reinvent_diversity = reinvent_diversity[reinvent_diversity["model"].isin(REINVENT_SELECTED_MODELS)].copy()
reinvent_distance = reinvent_distance[reinvent_distance["model"].isin(REINVENT_SELECTED_MODELS)].copy()
reinvent_distance["reference_label"] = reinvent_distance["reference"].map({"base": "Base", "good": "Liability-free reference"})

for col in range(3):
    ax = reinvent_fcd_ax if col == 2 else axes[4, col]
    if col < 2:
        bottom_specs = [
            (reinvent_diversity, "unique_scaffold_fraction", "Unique scaffold fraction", "percent"),
            (reinvent_diversity, "pairwise_tanimoto_distance_mean", "Pairwise Tanimoto distance", "decimal"),
        ]
        frame, metric, title, scale = bottom_specs[col]
        sns.boxplot(
            data=frame,
            x="method_label",
            y=metric,
            order=[REINVENT_LABELS[m] for m in REINVENT_SELECTED_MODELS],
            palette=[REINVENT_COLORS[m] for m in REINVENT_SELECTED_MODELS],
            width=0.58,
            linewidth=0.5,
            fliersize=0,
            ax=ax,
        )
        if scale == "percent":
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
    else:
        title = "FCD to reference"
        paired_reference_boxplot(
            ax, reinvent_distance, model_col="model", value_col="fcd", reference_col="reference_label",
            model_order=REINVENT_SELECTED_MODELS, model_labels=REINVENT_LABELS, model_colors=REINVENT_COLORS,
            reference_order=["Base", "Liability-free reference"],
            reference_labels={"Base": "Raw base", "Liability-free reference": "Liability-free reference"},
        )
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("")
    structure_ylabels = {
        0: "Scaffold fraction",
        1: "Tanimoto distance",
        2: "FCD",
    }
    ax.set_ylabel(structure_ylabels[col])
    if col < 2:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    style_ax(ax)

r_handles, r_labels = axes[2, 0].get_legend_handles_labels()
r_legend = dict(zip(r_labels, r_handles))
r_legend_order = ["Base", "Random NE", "Positive FT", "NE"]
fig.legend(
    [r_legend[label] for label in r_legend_order], r_legend_order,
    loc="lower center", bbox_to_anchor=(0.5, 1.015), ncol=4, frameon=False,
)
reinvent_panel_axes = [axes[row, col] for row in range(4) for col in range(4)]
reinvent_panel_axes.extend([axes[4, 0], axes[4, 1], reinvent_fcd_ax])
label_subplots(fig, axes=reinvent_panel_axes)
save_figure(fig, "figure_4_reinvent_liability")
plt.show()
endpoint_reinvent.to_csv(DATA_DIR / "figure_4_reinvent_endpoint.csv", index=False)
lambda_data.to_csv(DATA_DIR / "figure_4_reinvent_lambda_response.csv", index=False)
reinvent_diversity.to_csv(DATA_DIR / "figure_4_reinvent_chelator_internal_diversity.csv", index=False)
reinvent_distance.to_csv(DATA_DIR / "figure_4_reinvent_chelator_distribution_distance.csv", index=False)


## Figure 6: SemlaFlow 3D Liability Removal And Sampling Efficiency

Main claim: uncorrected full-model NE removes the joint liability set but damages 3D generation quality. Corrected NE directions retain PoseBusters performance, increase fixed-budget 3D usable yield, and produce more distinct usable scaffolds per sampling budget. Chemical-space displacement is shown against both the raw base generator and its liability-free subset. Detailed structural-diversity metrics, per-family liability rates, and paired statistics remain SI material.


In [ ]:
semlaflow_root = RESULTS / "external" / "semlaflow" / "four_liability_joint_replicates" / "analysis"
semlaflow_paper_dir = semlaflow_root / "paper_analysis"
semlaflow_scaffold_dir = semlaflow_root / "usable_scaffolds"

required_paths = {
    "2D endpoints": semlaflow_root / "replicate_metrics.csv",
    "PoseBusters endpoints": semlaflow_root / "posebusters_replicate_metrics.csv",
    "usable scaffolds": semlaflow_scaffold_dir / "usable_scaffold_metrics.csv",
    "FCD": semlaflow_root / "distribution_distance_fcd" / "distribution_distance_metrics.csv",
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing SemlaFlow analysis outputs:\n" + "\n".join(missing))

semla_endpoints = pd.read_csv(required_paths["2D endpoints"])
semla_pose = pd.read_csv(required_paths["PoseBusters endpoints"])
semla_scaffolds = pd.read_csv(required_paths["usable scaffolds"])
semla_fcd = pd.read_csv(required_paths["FCD"])
semla_endpoints = semla_endpoints[semla_endpoints["seed"].isin(PAPER_SEEDS_10)].copy()
semla_pose = semla_pose[semla_pose["seed"].isin(PAPER_SEEDS_10)].copy()
semla_scaffolds = semla_scaffolds[semla_scaffolds["seed"].isin(PAPER_SEEDS_10)].copy()
semla_fcd = semla_fcd[semla_fcd["seed"].isin(PAPER_SEEDS_10)].copy()

SEMLA_POSE_MODEL_MAP = {
    "base": "base",
    "random_tuned": "random_tuned",
    "positive_tuned": "positive_tuned",
    "bad_tuned": "bad_tuned",
    "standard_ne_2p5": "full_model_neon_lambda_2p5",
    "random_ne_2p5": "full_model_random_neon_lambda_2p5",
    "norm_matched_random_corrected_ne_2p5": "full_model_norm_matched_random_corrected_neon_lambda_2p5",
    "positive_corrected_ne_2p5": "full_model_positive_corrected_neon_lambda_2p5",
    "standard_ne_4": "full_model_neon_lambda_4",
    "random_ne_4": "full_model_random_neon_lambda_4",
    "norm_matched_random_corrected_ne_4": "full_model_norm_matched_random_corrected_neon_lambda_4",
    "positive_corrected_ne_4": "full_model_positive_corrected_neon_lambda_4",
}
semla_pose["model"] = semla_pose["model"].map(SEMLA_POSE_MODEL_MAP).fillna(semla_pose["model"])

SEMLA_METHOD_ORDER = [
    "base",
    "full_model_random_neon_lambda_2p5",
    "positive_tuned",
    "full_model_neon_lambda_2p5",
    "full_model_norm_matched_random_corrected_neon_lambda_2p5",
    "full_model_positive_corrected_neon_lambda_2p5",
]
SEMLA_VIABLE_ORDER = [
    "base",
    "positive_tuned",
    "full_model_neon_lambda_2p5",
    "full_model_positive_corrected_neon_lambda_2p5",
]
SEMLA_FCD_ORDER = SEMLA_VIABLE_ORDER.copy()
SEMLA_LABELS = {
    "base": "Base",
    "full_model_random_neon_lambda_2p5": "Random NE",
    "positive_tuned": "Positive FT",
    "full_model_neon_lambda_2p5": "Standard NE",
    "full_model_norm_matched_random_corrected_neon_lambda_2p5": "Random-corrected NE",
    "full_model_positive_corrected_neon_lambda_2p5": "Positive-corrected NE",
}
SEMLA_COLORS = {
    "base": METHOD_COLORS["base"],
    "full_model_random_neon_lambda_2p5": METHOD_COLORS["random_neon"],
    "positive_tuned": METHOD_COLORS["positive"],
    "full_model_neon_lambda_2p5": NE_LAMBDA_COLORS[0.50],
    "full_model_norm_matched_random_corrected_neon_lambda_2p5": "#52B69A",
    "full_model_positive_corrected_neon_lambda_2p5": METHOD_COLORS["positive_corrected_ne"],
}

def semla_boxplot(ax, frame, metric, order, ylabel, *, percent=False):
    plot = frame[frame["model"].isin(order)].copy()
    if percent:
        plot[metric] = 100.0 * plot[metric]
    sns.boxplot(
        data=plot,
        x="model",
        hue="model",
        y=metric,
        order=order,
        hue_order=order,
        palette=SEMLA_COLORS,
        dodge=False,
        legend=False,
        width=0.50,
        linewidth=0.8,
        fliersize=0,
        ax=ax,
    )
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_xticks(
        np.arange(len(order)),
        labels=[SEMLA_LABELS[model] for model in order],
        rotation=30,
        ha="right",
    )
    style_ax(ax)

fig = plt.figure(figsize=(14.2, 7.2), constrained_layout=True)
grid = fig.add_gridspec(2, 4)
ax_liability = fig.add_subplot(grid[0, 0])
ax_validity = fig.add_subplot(grid[0, 1])
ax_posebusters = fig.add_subplot(grid[0, 2])
ax_usable_3d = fig.add_subplot(grid[0, 3])
ax_fcd = fig.add_subplot(grid[1, 0])
ax_scaffold_diversity = fig.add_subplot(grid[1, 1])
ax_scaffold_yield = fig.add_subplot(grid[1, 2])
ax_efficiency = fig.add_subplot(grid[1, 3])

semla_boxplot(
    ax_liability,
    semla_endpoints,
    "four_liability_hit_fraction",
    SEMLA_METHOD_ORDER,
    "Liability-hit fraction (%)",
    percent=True,
)
ax_liability.set_title("Liability removal", loc="left", fontweight="bold")
ax_liability.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_validity,
    semla_endpoints,
    "valid_fraction",
    SEMLA_METHOD_ORDER,
    "RDKit-valid graph yield (%)",
    percent=True,
)
ax_validity.set_title("2D graph validity", loc="left", fontweight="bold")
ax_validity.yaxis.set_major_locator(MaxNLocator(nbins=5))

semla_boxplot(
    ax_posebusters,
    semla_pose,
    "posebusters_all_checks_fraction_of_rdkit_valid",
    SEMLA_METHOD_ORDER,
    "PoseBusters pass among RDKit-valid (%)",
    percent=True,
)
ax_posebusters.set_title("Conditional 3D quality", loc="left", fontweight="bold")
ax_posebusters.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_usable_3d,
    semla_pose,
    "usable_3d_yield",
    SEMLA_METHOD_ORDER,
    "3D usable yield (%)",
    percent=True,
)
ax_usable_3d.set_title("Practical 3D yield", loc="left", fontweight="bold")
ax_usable_3d.yaxis.set_major_locator(MaxNLocator(integer=True))

scaffold_3d = semla_scaffolds[semla_scaffolds["scope"].eq("3d_usable")].copy()
semla_boxplot(
    ax_scaffold_diversity,
    scaffold_3d,
    "unique_scaffold_fraction_among_usable",
    SEMLA_VIABLE_ORDER,
    "Unique scaffolds / 3D-usable molecules (%)",
    percent=True,
)
ax_scaffold_diversity.set_title("Scaffold diversity", loc="left", fontweight="bold")
ax_scaffold_diversity.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_scaffold_yield,
    scaffold_3d,
    "unique_usable_scaffold_yield",
    SEMLA_VIABLE_ORDER,
    "Unique 3D-usable scaffolds / sampled (%)",
    percent=True,
)
ax_scaffold_yield.set_title("Usable scaffold yield", loc="left", fontweight="bold")
ax_scaffold_yield.yaxis.set_major_locator(MaxNLocator(integer=True))

semla_boxplot(
    ax_efficiency,
    scaffold_3d,
    "samples_to_fixed_scaffold_target",
    SEMLA_VIABLE_ORDER,
    "Samples to 2,500 3D-usable scaffolds",
)
ax_efficiency.set_title("Fixed-target efficiency", loc="left", fontweight="bold")
ax_efficiency.yaxis.set_major_locator(MaxNLocator(integer=True))

fcd_references = ["base", "liability_free_base"]
fcd_labels = {"base": "Raw base", "liability_free_base": "Liability-free base"}
fcd_plot = semla_fcd[
    semla_fcd["model"].isin(SEMLA_FCD_ORDER)
    & semla_fcd["reference"].isin(fcd_references)
].copy()
if fcd_plot["fcd"].isna().any():
    raise ValueError("SemlaFlow FCD table contains missing values")

paired_reference_boxplot(
    ax_fcd, fcd_plot, model_col="model", value_col="fcd", reference_col="reference",
    model_order=SEMLA_FCD_ORDER, model_labels=SEMLA_LABELS, model_colors=SEMLA_COLORS,
    reference_order=fcd_references, reference_labels=fcd_labels,
)
ax_fcd.set_ylabel("FCD to reference")
ax_fcd.set_title("Chemical-space shift", loc="left", fontweight="bold")

legend_handles = [
    Patch(
        facecolor=SEMLA_COLORS[model],
        edgecolor="#111111",
        linewidth=0.6,
        label=SEMLA_LABELS[model],
    )
    for model in SEMLA_METHOD_ORDER
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=6,
    frameon=False,
    bbox_to_anchor=(0.5, 1.025),
)
label_subplots(fig, x=-0.10, y=1.04)
save_figure(fig, "figure_6_semlaflow_3d_liability")
plt.show()

semla_endpoints[semla_endpoints["model"].isin(SEMLA_METHOD_ORDER)].to_csv(
    DATA_DIR / "figure_6_semlaflow_2d_endpoints.csv", index=False
)
semla_pose[semla_pose["model"].isin(SEMLA_METHOD_ORDER)].to_csv(
    DATA_DIR / "figure_6_semlaflow_posebusters_endpoints.csv", index=False
)
scaffold_3d[scaffold_3d["model"].isin(SEMLA_VIABLE_ORDER)].to_csv(
    DATA_DIR / "figure_6_semlaflow_usable_scaffolds.csv", index=False
)
fcd_plot.to_csv(DATA_DIR / "figure_6_semlaflow_fcd.csv", index=False)
